In [1]:
import pandas as pd
import vivarium_inputs
import vivarium.gbd_mapping as gbd_mapping
import pathlib
from lsff_utils import config_utils
from lsff_utils.results import expand_to_all_scenarios, aggregate_by_scenario

In [2]:
location = "india"
vehicle = "rice"

In [3]:
# Parameters
location = "india"
vehicle = "rice"


In [4]:
scenarios = list(
    config_utils.get_location_fortificant_vehicle_intervention_scenarios()
    .pipe(lambda df: df[(df.location == location) & (df.vehicle == vehicle)])
    .intervention_scenario.unique()
) + ["zero", "baseline"]
scenarios

['intervention', 'zero', 'baseline']

In [5]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/person_time_anemia.parquet"
if pathlib.Path(path).is_file():
    pregnancy_person_time_anemia = pd.read_parquet(path)
else:
    pregnancy_person_time_anemia = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/person_time_anemia.parquet"
        ).assign(value=0),
        scenarios,
    )
pregnancy_person_time_anemia

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,person_time,impairment,anemia,not_anemic,10_to_14,invalid,1,baseline,0,2,176.044657
1,person_time,impairment,anemia,not_anemic,10_to_14,invalid,2,baseline,0,2,153.002163
2,person_time,impairment,anemia,not_anemic,10_to_14,invalid,3,baseline,0,2,139.176666
3,person_time,impairment,anemia,not_anemic,10_to_14,invalid,4,baseline,0,2,150.237064
4,person_time,impairment,anemia,not_anemic,10_to_14,invalid,5,baseline,0,2,54.380287
...,...,...,...,...,...,...,...,...,...,...,...
53995,person_time,impairment,anemia,severe,95_plus,severe,1,baseline,0,9,0.000000
53996,person_time,impairment,anemia,severe,95_plus,severe,2,baseline,0,9,0.000000
53997,person_time,impairment,anemia,severe,95_plus,severe,3,baseline,0,9,0.000000
53998,person_time,impairment,anemia,severe,95_plus,severe,4,baseline,0,9,0.000000


In [6]:
pregnancy_person_time_anemia.groupby("scenario").random_seed.nunique()

scenario
baseline        10
intervention    10
zero            10
Name: random_seed, dtype: int64

In [7]:
pregnancy_person_time_anemia.sub_entity.value_counts()

not_anemic    13500
mild          13500
moderate      13500
severe        13500
Name: sub_entity, dtype: int64

In [8]:
total_pregnant_person_time = aggregate_by_scenario(pregnancy_person_time_anemia)
total_pregnant_person_time

scenario      wealth_quintile
baseline      1                  4.934299e+06
              2                  4.077383e+06
              3                  3.657170e+06
              4                  3.470089e+06
              5                  3.396811e+06
intervention  1                  4.934299e+06
              2                  4.077383e+06
              3                  3.657170e+06
              4                  3.470089e+06
              5                  3.396811e+06
zero          1                  4.934293e+06
              2                  4.077364e+06
              3                  3.657151e+06
              4                  3.470089e+06
              5                  3.396805e+06
Name: value, dtype: float64

In [9]:
anemic_pregnant_person_time = aggregate_by_scenario(
    pregnancy_person_time_anemia[
        pregnancy_person_time_anemia.sub_entity != "not_anemic"
    ]
)
anemic_pregnant_person_time

scenario      wealth_quintile
baseline      1                  2.579940e+06
              2                  2.033973e+06
              3                  1.729041e+06
              4                  1.534192e+06
              5                  1.294183e+06
intervention  1                  2.579940e+06
              2                  2.033973e+06
              3                  1.729041e+06
              4                  1.534192e+06
              5                  1.294183e+06
zero          1                  2.759666e+06
              2                  2.165960e+06
              3                  1.840347e+06
              4                  1.625970e+06
              5                  1.348687e+06
Name: value, dtype: float64

In [10]:
pregnant_anemia_prevalence_by_scenario = (
    anemic_pregnant_person_time / total_pregnant_person_time
).fillna(0)
pregnant_anemia_prevalence_by_scenario

scenario      wealth_quintile
baseline      1                  0.522858
              2                  0.498843
              3                  0.472781
              4                  0.442119
              5                  0.380999
intervention  1                  0.522858
              2                  0.498843
              3                  0.472781
              4                  0.442119
              5                  0.380999
zero          1                  0.559283
              2                  0.531216
              3                  0.503219
              4                  0.468567
              5                  0.397046
Name: value, dtype: float64

In [11]:
path = f"./results/{location}/{vehicle}/pregnant_anemia_prevalence_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
pregnant_anemia_prevalence_by_scenario.to_csv(path)

In [12]:
pop = pd.read_csv(f"../0100_data_prep/results/population/stratified/{location}.csv")
pop

,sex,age_start,age_end,pregnant,wealth_quintile,value
0,Female,0.0,0.019178,not_pregnant,1,48678.711021
1,Female,0.0,0.019178,not_pregnant,2,42723.422230
2,Female,0.0,0.019178,not_pregnant,3,37932.716572
3,Female,0.0,0.019178,not_pregnant,4,35728.167677
4,Female,0.0,0.019178,not_pregnant,5,28631.476092
...,...,...,...,...,...,...
280,Male,95.0,125.000000,not_pregnant,1,18328.880607
281,Male,95.0,125.000000,not_pregnant,2,19140.234171
282,Male,95.0,125.000000,not_pregnant,3,19770.250303
283,Male,95.0,125.000000,not_pregnant,4,20851.307187


In [13]:
pregnant_pop = pop[pop.pregnant == "pregnant"].groupby(["wealth_quintile"]).value.sum()
pregnant_pop

wealth_quintile
1    4.478246e+06
2    3.707329e+06
3    3.317141e+06
4    3.157155e+06
5    3.080667e+06
Name: value, dtype: float64

In [14]:
pregnancy_prevalent_anemia_cases_by_scenario = (
    pregnant_anemia_prevalence_by_scenario * pregnant_pop
)
pregnancy_prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  2.341489e+06
              2                  1.849374e+06
              3                  1.568282e+06
              4                  1.395838e+06
              5                  1.173732e+06
intervention  1                  2.341489e+06
              2                  1.849374e+06
              3                  1.568282e+06
              4                  1.395838e+06
              5                  1.173732e+06
zero          1                  2.504607e+06
              2                  1.969392e+06
              3                  1.669248e+06
              4                  1.479339e+06
              5                  1.223166e+06
Name: value, dtype: float64

In [15]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/transition_count_maternal_disorders.parquet"
if pathlib.Path(path).is_file():
    maternal_disorders_transition_counts = pd.read_parquet(path)
else:
    maternal_disorders_transition_counts = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/transition_count_maternal_disorders.parquet"
        ).assign(value=0),
        scenarios,
    )

maternal_disorders_transition_counts

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,1,baseline,0,2,0.0
1,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,2,baseline,0,2,0.0
2,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,3,baseline,0,2,0.0
3,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,4,baseline,0,2,0.0
4,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,5,baseline,0,2,0.0
...,...,...,...,...,...,...,...,...,...,...,...
26995,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,1,baseline,0,9,0.0
26996,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,2,baseline,0,9,0.0
26997,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,3,baseline,0,9,0.0
26998,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,4,baseline,0,9,0.0


In [16]:
maternal_disorders_transition_counts.sub_entity.cat.categories

Index(['susceptible_to_maternal_disorders_to_maternal_disorders', 'maternal_disorders_to_recovered_from_maternal_disorders'], dtype='object')

In [17]:
maternal_disorders_incident_cases_by_scenario = aggregate_by_scenario(
    maternal_disorders_transition_counts[
        maternal_disorders_transition_counts.sub_entity
        == "susceptible_to_maternal_disorders_to_maternal_disorders"
    ]
)
maternal_disorders_incident_cases_by_scenario

scenario      wealth_quintile
baseline      1                  4.061404e+06
              2                  2.434418e+06
              3                  2.923813e+06
              4                  2.532288e+06
              5                  1.525221e+06
intervention  1                  4.061404e+06
              2                  2.434418e+06
              3                  2.923813e+06
              4                  2.532288e+06
              5                  1.525221e+06
zero          1                  4.146721e+06
              2                  2.492611e+06
              3                  2.977965e+06
              4                  2.573311e+06
              5                  1.541765e+06
Name: value, dtype: float64

In [18]:
path = (
    f"./results/{location}/{vehicle}/maternal_disorders_incident_cases_by_scenario.csv"
)
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
maternal_disorders_incident_cases_by_scenario.to_csv(path)

In [19]:
path = f"results/rescaled_child_results/{vehicle}/{location}/deaths.parquet"
if pathlib.Path(path).is_file():
    neonatal_deaths = pd.read_parquet(path).rename(
        columns={"maternal_scenario": "scenario"}
    )
else:
    neonatal_deaths = expand_to_all_scenarios(
        pd.read_parquet(f"results/rescaled_child_results/rice/india/deaths.parquet")
        .assign(value=0)
        .rename(columns={"maternal_scenario": "scenario"}),
        scenarios,
    )

neonatal_deaths

,measure,entity_type,entity,sub_entity,age_group,sex,wealth_quintile,child_scenario,scenario,input_draw,random_seed,value
0,deaths,cause,other_causes,other_causes,0_to_5_months,Female,1,baseline,intervention,0,2,7125.308488
1,deaths,cause,other_causes,other_causes,0_to_5_months,Female,2,baseline,intervention,0,2,5440.269319
2,deaths,cause,other_causes,other_causes,0_to_5_months,Female,3,baseline,intervention,0,2,5392.125343
3,deaths,cause,other_causes,other_causes,0_to_5_months,Female,4,baseline,intervention,0,2,5006.973532
4,deaths,cause,other_causes,other_causes,0_to_5_months,Female,5,baseline,intervention,0,2,5199.549437
...,...,...,...,...,...,...,...,...,...,...,...,...
1195,deaths,cause,other_causes,other_causes,18_to_59_months,Male,1,baseline,intervention,0,6,914.735549
1196,deaths,cause,other_causes,other_causes,18_to_59_months,Male,2,baseline,intervention,0,6,722.159644
1197,deaths,cause,other_causes,other_causes,18_to_59_months,Male,3,baseline,intervention,0,6,625.871692
1198,deaths,cause,other_causes,other_causes,18_to_59_months,Male,4,baseline,intervention,0,6,385.151810


In [20]:
neonatal_deaths_by_scenario = aggregate_by_scenario(neonatal_deaths)
neonatal_deaths_by_scenario

scenario      wealth_quintile
baseline      1                  191516.737612
              2                  159549.137367
              3                  144046.777008
              4                  131047.903414
              5                  129507.296173
intervention  1                  191516.737612
              2                  159549.137367
              3                  144046.777008
              4                  131047.903414
              5                  129507.296173
zero          1                  191901.889422
              2                  159452.849415
              3                  144576.360747
              4                  131096.047390
              5                  129651.728102
Name: value, dtype: float64

In [21]:
path = f"./results/{location}/{vehicle}/neonatal_deaths_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
neonatal_deaths_by_scenario.to_csv(path)

In [22]:
path = f"../0400_non_pregnant_anemia_model/results/{vehicle}/{location}/anemia_cases.parquet"
if pathlib.Path(path).is_file():
    non_pregnancy_anemia_cases = pd.read_parquet(path)
else:
    non_pregnancy_anemia_cases = expand_to_all_scenarios(
        pd.read_parquet(
            f"../0400_non_pregnant_anemia_model/results/rice/india/anemia_cases.parquet"
        ).assign(value=0),
        scenarios,
    )

non_pregnancy_anemia_cases

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,1,39601.769161,zero
1,Female,0.0,0.019178,2,32639.498130,zero
2,Female,0.0,0.019178,3,28574.791970,zero
3,Female,0.0,0.019178,4,25160.080603,zero
4,Female,0.0,0.019178,5,18957.590376,zero
...,...,...,...,...,...,...
745,Male,95.0,125.000000,1,8666.102516,intervention
746,Male,95.0,125.000000,2,8217.835776,intervention
747,Male,95.0,125.000000,3,8273.215336,intervention
748,Male,95.0,125.000000,4,8136.541097,intervention


In [23]:
non_pregnancy_prevalent_anemia_cases_by_scenario = aggregate_by_scenario(
    non_pregnancy_anemia_cases.assign(entity="anemia", input_draw="draw_0")
)
non_pregnancy_prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  1.224695e+08
              2                  1.150335e+08
              3                  1.138088e+08
              4                  1.086529e+08
              5                  1.031774e+08
intervention  1                  1.224695e+08
              2                  1.150335e+08
              3                  1.138088e+08
              4                  1.086529e+08
              5                  1.031774e+08
zero          1                  1.291309e+08
              2                  1.213341e+08
              3                  1.193293e+08
              4                  1.134219e+08
              5                  1.058060e+08
Name: value, dtype: float64

In [24]:
prevalent_anemia_cases_by_scenario = (
    pregnancy_prevalent_anemia_cases_by_scenario
    + non_pregnancy_prevalent_anemia_cases_by_scenario
)
prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  1.248110e+08
              2                  1.168829e+08
              3                  1.153771e+08
              4                  1.100488e+08
              5                  1.043511e+08
intervention  1                  1.248110e+08
              2                  1.168829e+08
              3                  1.153771e+08
              4                  1.100488e+08
              5                  1.043511e+08
zero          1                  1.316355e+08
              2                  1.233035e+08
              3                  1.209986e+08
              4                  1.149012e+08
              5                  1.070292e+08
Name: value, dtype: float64

In [25]:
path = f"./results/{location}/{vehicle}/prevalent_anemia_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
prevalent_anemia_cases_by_scenario.to_csv(path)

In [26]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/ntd_cases_by_scenario.csv"
if pathlib.Path(path).is_file():
    ntd_cases_by_scenario = pd.read_csv(path)
else:
    ntd_cases_by_scenario = expand_to_all_scenarios(
        pd.read_csv(
            f"../0500_neural_tube_defects_model/results/india/rice/ntd_cases_by_scenario.csv"
        ).assign(value=0),
        scenarios,
    )

ntd_cases_by_scenario = ntd_cases_by_scenario.set_index(
    ["scenario", "wealth_quintile"]
).value
ntd_cases_by_scenario

scenario      wealth_quintile
zero          1                  5246.952233
              2                  4635.134710
              3                  4161.261596
              4                  3923.864610
              5                  3341.790775
baseline      1                  4876.704553
              2                  4356.153811
              3                  3949.449160
              4                  3750.255554
              5                  3275.585263
intervention  1                  2765.653402
              2                  2658.591122
              3                  2580.494071
              4                  2577.117613
              5                  2743.103995
Name: value, dtype: float64

In [27]:
path = f"./results/{location}/{vehicle}/ntd_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ntd_cases_by_scenario.to_csv(path)